In [2]:
#kernel thesis clean4
import pickle
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_pickle("screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [8]:
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis]  
y_data = np.array(df['class_values'].tolist())
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

x_data = np.transpose(x_data, (0, 2, 1))  #reshaped to (num_samples, num_features, sequence_length)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full,y_train_full,test_size=0.25,stratify=y_train_full,random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: torch.Size([7500, 1, 800]) torch.Size([7500])
Validation: torch.Size([2500, 1, 800]) torch.Size([2500])
Test: torch.Size([2500, 1, 800]) torch.Size([2500])


In [4]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [6]:
import torch
import torch.nn as nn
import sys
sys.path.append(r"C:\Users\Patrick\InceptionTime-Pytorch")
from inception import InceptionBlock


class Flatten(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x.mean(-1)   # safer als view


model = nn.Sequential(
    InceptionBlock(
        in_channels=1,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)

In [15]:
from sklearn.metrics import f1_score
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

earlystop = EarlyStopper(patience=20, min_delta=0.001)

epochs = 50

train_losses = []
val_losses = []
val_f1_scores = []

best_val_f1 = -np.inf
best_model_state = None

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    model.eval()

    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_val_batch, y_val_batch in val_loader:
            X_val_batch = X_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            outputs = model(X_val_batch)
            loss = criterion(outputs, y_val_batch)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_val_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_f1_scores.append(val_f1)
    scheduler.step(avg_val_loss)

    #best model speichern für test
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()

    if earlystop.early_stop(avg_val_loss):
        print("Early stopping triggered")
        break

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1 Score: {val_f1:.4f}")


Epoch 1/50, Train Loss: 1.8144, Val Loss: 1.6851, Val F1 Score: 0.2369
Epoch 2/50, Train Loss: 1.6158, Val Loss: 1.9331, Val F1 Score: 0.1399
Epoch 3/50, Train Loss: 1.5667, Val Loss: 3.0216, Val F1 Score: 0.1762
Epoch 4/50, Train Loss: 1.5267, Val Loss: 1.7450, Val F1 Score: 0.1951
Epoch 5/50, Train Loss: 1.5003, Val Loss: 3.5873, Val F1 Score: 0.1475
Epoch 6/50, Train Loss: 1.4510, Val Loss: 1.8282, Val F1 Score: 0.2610
Epoch 7/50, Train Loss: 1.4281, Val Loss: 2.5260, Val F1 Score: 0.2225
Epoch 8/50, Train Loss: 1.4097, Val Loss: 2.1014, Val F1 Score: 0.1899
Epoch 9/50, Train Loss: 1.3977, Val Loss: 1.9748, Val F1 Score: 0.2471
Epoch 10/50, Train Loss: 1.3536, Val Loss: 2.0963, Val F1 Score: 0.2010
Epoch 11/50, Train Loss: 1.3256, Val Loss: 1.4083, Val F1 Score: 0.3853
Epoch 12/50, Train Loss: 1.3090, Val Loss: 1.9746, Val F1 Score: 0.2039
Epoch 13/50, Train Loss: 1.2941, Val Loss: 1.6695, Val F1 Score: 0.2487
Epoch 14/50, Train Loss: 1.2626, Val Loss: 1.5353, Val F1 Score: 0.3214
E

In [17]:
model.load_state_dict(best_model_state)
torch.save(best_model_state, "best_inception_model.pth")
model.eval()

test_preds = []
test_labels = []
test_loss = 0.0

with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        outputs = model(X_test_batch)
        loss = criterion(outputs, y_test_batch)

        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

Test Loss: 1.1346
Test F1 Macro: 0.4961


## 3-CV InceptionTime

In [10]:
from sklearn.metrics import f1_score
import math
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis]
y_data = np.array(df['class_values'].tolist())
print("x_data shape:", x_data.shape)
x_data = np.transpose(x_data, (0, 2, 1))
print("transposed x_data shape:", x_data.shape)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=81)

cv_scores = []

best_overall_model_state = None
best_overall_f1 = -np.inf


for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):

    print(f"Fold {fold+1}")

    X_train_fold = X_train_full[train_idx]
    y_train_fold = y_train_full[train_idx]

    X_val_fold = X_train_full[val_idx]
    y_val_fold = y_train_full[val_idx]

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32),torch.tensor(y_train_fold, dtype=torch.long)),batch_size=32,shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32),torch.tensor(y_val_fold, dtype=torch.long)),batch_size=32,shuffle=False)
    model = nn.Sequential(
    InceptionBlock(
        in_channels=1,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    earlystop = EarlyStopper(patience=7, min_delta=0.001)

    epochs = 50

    best_val_f1 = -np.inf
    best_model_state = None


    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                X_val_batch = X_val_batch.to(device)
                y_val_batch = y_val_batch.to(device)

                outputs = model(X_val_batch)
                loss = criterion(outputs, y_val_batch)

                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_val_batch.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        scheduler.step(avg_val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()

        if earlystop.early_stop(avg_val_loss):
            print("Early stopping triggered")
            break

        print(f"Fold {fold+1}, Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")

    cv_scores.append(best_val_f1)
    if best_val_f1 > best_overall_f1:
        best_overall_f1 = best_val_f1
        best_overall_model_state = best_model_state


print(f"CV f1 mean avg score: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")

x_data shape: (12500, 800, 1)
transposed x_data shape: (12500, 1, 800)
Fold 1
Fold 1, Epoch 1/50, Train Loss: 1.8443, Val Loss: 1.9652, Val F1: 0.1715
Fold 1, Epoch 2/50, Train Loss: 1.6583, Val Loss: 1.7995, Val F1: 0.1985
Fold 1, Epoch 3/50, Train Loss: 1.5888, Val Loss: 3.8989, Val F1: 0.1323
Fold 1, Epoch 4/50, Train Loss: 1.5576, Val Loss: 1.7108, Val F1: 0.2837
Fold 1, Epoch 5/50, Train Loss: 1.5215, Val Loss: 3.3040, Val F1: 0.1134
Fold 1, Epoch 6/50, Train Loss: 1.4999, Val Loss: 2.0547, Val F1: 0.1621
Fold 1, Epoch 7/50, Train Loss: 1.4831, Val Loss: 1.9992, Val F1: 0.2217
Fold 1, Epoch 8/50, Train Loss: 1.4459, Val Loss: 1.7045, Val F1: 0.2633
Fold 1, Epoch 9/50, Train Loss: 1.4458, Val Loss: 2.1879, Val F1: 0.2220
Fold 1, Epoch 10/50, Train Loss: 1.4276, Val Loss: 4.3113, Val F1: 0.1996
Fold 1, Epoch 11/50, Train Loss: 1.4113, Val Loss: 2.5541, Val F1: 0.1366
Fold 1, Epoch 12/50, Train Loss: 1.3763, Val Loss: 1.4650, Val F1: 0.3363
Fold 1, Epoch 13/50, Train Loss: 1.3600, Va

In [11]:
model = nn.Sequential(
    InceptionBlock(
        in_channels=1,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)
final_model = model
final_model.to(device)
final_model.load_state_dict(best_overall_model_state)
final_model.eval()

test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),torch.tensor(y_test, dtype=torch.long)),batch_size=32,shuffle=False)

all_preds = []
all_labels = []
test_loss = 0.0

criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:

        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)
        outputs = final_model(X_test_batch)
        loss = criterion(outputs, y_test_batch)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"Final Test F1 Macro: {test_f1:.4f}")

Final Test F1 Macro: 0.5172
